In [1]:
# ==========================================================
# STUDENT 2
# NOTEBOOK 06
# ROBERTA FINE-TUNING
# ==========================================================

import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup
)

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

In [2]:
# ==========================================================
# CONFIGURATION
# ==========================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent

PROCESSED_DATA = PROJECT_ROOT / "datasets" / "processed"

MODELS = PROJECT_ROOT / "models"

REPORTS = PROJECT_ROOT / "reports"

FIGURES = PROJECT_ROOT / "figures"

MODEL_NAME = "roberta-base"

MAX_LENGTH = 512

BATCH_SIZE = 8

EPOCHS = 3

LEARNING_RATE = 2e-5

print("="*70)
print("ROBERTA CONFIGURATION")
print("="*70)

print("Device:", device)
print("Model :", MODEL_NAME)

ROBERTA CONFIGURATION
Device: cpu
Model : roberta-base


In [3]:
train_df = pd.read_csv(
    PROCESSED_DATA / "bert_train_subset.csv"
)

validation_df = pd.read_csv(
    PROCESSED_DATA / "bert_validation_subset.csv"
)

test_df = pd.read_csv(
    PROCESSED_DATA / "bert_test_subset.csv"
)

print(train_df.shape)
print(validation_df.shape)
print(test_df.shape)

(5000, 7)
(1000, 7)
(1000, 7)


In [4]:
# ==========================================================
# EMAIL DATASET
# ==========================================================

class EmailDataset(Dataset):

    def __init__(self, dataframe, tokenizer):

        self.data = dataframe.reset_index(drop=True)

        self.tokenizer = tokenizer

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        text = str(
            self.data.loc[idx, "cleaned_email_text"]
        )

        label = int(
            self.data.loc[idx, "label"]
        )

        encoding = self.tokenizer(

            text,

            truncation=True,

            max_length=MAX_LENGTH,

            padding=False,

            return_tensors="pt"

        )

        item = {

            "input_ids":
                encoding["input_ids"].squeeze(0),

            "attention_mask":
                encoding["attention_mask"].squeeze(0),

            "labels":
                torch.tensor(
                    label,
                    dtype=torch.long
                )

        }

        return item

print("="*70)
print("EmailDataset Ready")
print("="*70)

EmailDataset Ready


In [6]:
# ==========================================================
# LOAD ROBERTA TOKENIZER
# ==========================================================

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("=" * 70)
print("ROBERTA TOKENIZER LOADED")
print("=" * 70)

print(tokenizer.__class__.__name__)
print("Vocabulary Size :", tokenizer.vocab_size)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

ROBERTA TOKENIZER LOADED
RobertaTokenizer
Vocabulary Size : 50265


In [7]:
# ==========================================================
# CREATE DATALOADERS
# ==========================================================

collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

train_dataset = EmailDataset(
    train_df,
    tokenizer
)

validation_dataset = EmailDataset(
    validation_df,
    tokenizer
)

test_dataset = EmailDataset(
    test_df,
    tokenizer
)

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    collate_fn=collator

)

validation_loader = DataLoader(

    validation_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    collate_fn=collator

)

test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    collate_fn=collator

)

print("="*70)
print("DATALOADERS CREATED")
print("="*70)

print("Training batches   :", len(train_loader))
print("Validation batches :", len(validation_loader))
print("Test batches       :", len(test_loader))

DATALOADERS CREATED
Training batches   : 625
Validation batches : 125
Test batches       : 125


In [8]:
# ==========================================================
# LOAD ROBERTA MODEL
# ==========================================================

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=2

)

model.to(device)

print("=" * 70)
print("ROBERTA MODEL LOADED")
print("=" * 70)

print(type(model).__name__)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ROBERTA MODEL LOADED
RobertaForSequenceClassification


In [9]:
# ==========================================================
# OPTIMIZER AND SCHEDULER
# ==========================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LEARNING_RATE

)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(

    optimizer,

    num_warmup_steps=0,

    num_training_steps=total_steps

)

print("=" * 70)
print("OPTIMIZER READY")
print("=" * 70)

print("Total Steps :", total_steps)

OPTIMIZER READY
Total Steps : 1875


In [10]:
# ==========================================================
# CLASS WEIGHTS
# ==========================================================

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(train_df["label"])

weights = compute_class_weight(

    class_weight="balanced",

    classes=classes,

    y=train_df["label"]

)

class_weights = torch.tensor(

    weights,

    dtype=torch.float

).to(device)

print("=" * 70)
print("CLASS WEIGHTS")
print("=" * 70)

print(class_weights)

CLASS WEIGHTS
tensor([0.5492, 5.5804])


In [11]:
# ==========================================================
# WEIGHTED LOSS FUNCTION
# ==========================================================

criterion = torch.nn.CrossEntropyLoss(

    weight=class_weights

)

print("=" * 70)
print("LOSS FUNCTION READY")
print("=" * 70)

LOSS FUNCTION READY


In [12]:
# ==========================================================
# TRAINING HISTORY
# ==========================================================

import time

train_losses = []
validation_losses = []

train_accuracies = []
validation_accuracies = []

epoch_training_times = []
epoch_validation_times = []

best_validation_loss = float("inf")

patience = 2
counter = 0

print("=" * 70)
print("TRAINING HISTORY INITIALIZED")
print("=" * 70)

TRAINING HISTORY INITIALIZED


In [13]:
# ==========================================================
# ROBERTA TRAINING LOOP
# ==========================================================

training_start = time.time()

for epoch in range(EPOCHS):

    print("\n" + "=" * 70)
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print("=" * 70)

    # ------------------------------------------------------
    # TRAINING
    # ------------------------------------------------------

    model.train()

    train_start = time.time()

    running_loss = 0

    correct = 0

    total = 0

    for batch in tqdm(train_loader, desc="Training"):

        input_ids = batch["input_ids"].to(device)

        attention_mask = batch["attention_mask"].to(device)

        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(

            input_ids=input_ids,

            attention_mask=attention_mask

        )

        logits = outputs.logits

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        scheduler.step()

        running_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        correct += (preds == labels).sum().item()

        total += labels.size(0)

    train_loss = running_loss / len(train_loader)

    train_accuracy = correct / total

    train_end = time.time()

    # ------------------------------------------------------
    # VALIDATION
    # ------------------------------------------------------

    model.eval()

    validation_start = time.time()

    running_validation_loss = 0

    correct = 0

    total = 0

    with torch.no_grad():

        for batch in tqdm(validation_loader, desc="Validation"):

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].to(device)

            outputs = model(

                input_ids=input_ids,

                attention_mask=attention_mask

            )

            logits = outputs.logits

            loss = criterion(
                logits,
                labels
            )

            running_validation_loss += loss.item()

            preds = torch.argmax(logits, dim=1)

            correct += (preds == labels).sum().item()

            total += labels.size(0)

    validation_loss = running_validation_loss / len(validation_loader)

    validation_accuracy = correct / total

    validation_end = time.time()

    train_losses.append(train_loss)
    validation_losses.append(validation_loss)

    train_accuracies.append(train_accuracy)
    validation_accuracies.append(validation_accuracy)

    epoch_training_times.append(train_end - train_start)
    epoch_validation_times.append(validation_end - validation_start)

    print(f"\nTrain Loss            : {train_loss:.4f}")
    print(f"Train Accuracy        : {train_accuracy:.4f}")
    print(f"Validation Loss       : {validation_loss:.4f}")
    print(f"Validation Accuracy   : {validation_accuracy:.4f}")

    print(f"Training Time (sec)   : {train_end-train_start:.2f}")
    print(f"Validation Time (sec) : {validation_end-validation_start:.2f}")

    if validation_loss < best_validation_loss:

        best_validation_loss = validation_loss

        counter = 0

        torch.save(

            model.state_dict(),

            MODELS / "best_roberta_model.pt"

        )

        print("✓ Best model saved.")

    else:

        counter += 1

        print(f"No improvement ({counter}/{patience})")

        if counter >= patience:

            print("Early stopping triggered.")

            break

training_end = time.time()

total_training_time = training_end - training_start

print("\n" + "=" * 70)
print("TRAINING FINISHED")
print("=" * 70)

print(f"Total Training Time : {total_training_time/60:.2f} minutes")


Epoch 1/3


Training:   0%|          | 0/625 [00:00<?, ?it/s]

Validation:   0%|          | 0/125 [00:01<?, ?it/s]


Train Loss            : 0.1631
Train Accuracy        : 0.9606
Validation Loss       : 0.1180
Validation Accuracy   : 0.9910
Training Time (sec)   : 21012.16
Validation Time (sec) : 1033.12
✓ Best model saved.

Epoch 2/3


Training:   0%|          | 0/625 [00:00<?, ?it/s]

Validation:   0%|          | 0/125 [00:06<?, ?it/s]


Train Loss            : 0.0518
Train Accuracy        : 0.9874
Validation Loss       : 0.1333
Validation Accuracy   : 0.9900
Training Time (sec)   : 20558.40
Validation Time (sec) : 945.61
No improvement (1/2)

Epoch 3/3


Training:   0%|          | 0/625 [00:00<?, ?it/s]

Validation:   0%|          | 0/125 [00:01<?, ?it/s]


Train Loss            : 0.0138
Train Accuracy        : 0.9970
Validation Loss       : 0.1580
Validation Accuracy   : 0.9920
Training Time (sec)   : 22443.37
Validation Time (sec) : 945.32
No improvement (2/2)
Early stopping triggered.

TRAINING FINISHED
Total Training Time : 1115.82 minutes
